In [0]:
taxi_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2009-01.csv.gz")

In [0]:
taxi_df.columns

Out[7]: ['vendor_name',
 'Trip_Pickup_DateTime',
 'Trip_Dropoff_DateTime',
 'Passenger_Count',
 'Trip_Distance',
 'Start_Lon',
 'Start_Lat',
 'Rate_Code',
 'store_and_forward',
 'End_Lon',
 'End_Lat',
 'Payment_Type',
 'Fare_Amt',
 'surcharge',
 'mta_tax',
 'Tip_Amt',
 'Tolls_Amt',
 'Total_Amt']

In [0]:

pickup_longitude = "Start_Lon"
pickup_latitude = "Start_Lat"
dropoff_longitude = "End_Lon"
dropoff_latitude = "End_Lat"

In [0]:
from pyspark.sql.functions import expr

taxi_geo = taxi_df \
    .filter(f"{pickup_longitude} IS NOT NULL AND {pickup_latitude} IS NOT NULL") \
    .withColumn("pickup_point", expr(f"ST_Point(cast({pickup_longitude} as double), cast({pickup_latitude} as double))")) \
    .withColumn("dropoff_point", expr(f"ST_Point(cast({dropoff_longitude} as double), cast({dropoff_latitude} as double))"))

taxi_geo.createOrReplaceTempView("taxi_geo")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-1297142883199437>:3
      1 from pyspark.sql.functions import expr
----> 3 taxi_geo = taxi_df \
      4     .filter(f"{pickup_longitude} IS NOT NULL AND {pickup_latitude} IS NOT NULL") \
      5     .withColumn("pickup_point", expr(f"ST_Point(cast({pickup_longitude} as double), cast({pickup_latitude} as double))")) \
      6     .withColumn("dropoff_point", expr(f"ST_Point(cast({dropoff_longitude} as double), cast({dropoff_latitude} as double))"))
      8 taxi_geo.createOrReplaceTempView("taxi_geo")

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, sig

In [0]:
polygon_wkt = "POLYGON((-74.02 40.70, -74.02 40.88, -73.93 40.88, -73.93 40.70, -74.02 40.70))"
polygon_df = spark.sql(f"SELECT ST_GeomFromText('{polygon_wkt}') AS polygon")
polygon_df.createOrReplaceTempView("polygon_table")

In [0]:
%sql
SELECT COUNT(*) AS pickups_in_polygon
FROM taxi_geo, polygon_table
WHERE ST_Contains(polygon, pickup_point)

In [0]:
%sql
SELECT COUNT(*) AS pickups_within
FROM taxi_geo, polygon_table
WHERE ST_Within(pickup_point, polygon)

In [0]:
%sql
SELECT COUNT(*) AS pickups_intersects
FROM taxi_geo, polygon_table
WHERE ST_Intersects(pickup_point, polygon)

In [0]:
%sql
SELECT ST_Distance(pickup_point, dropoff_point) AS trip_distance
FROM taxi_geo
LIMIT 10

In [0]:
%sql
SELECT ST_Area(polygon) AS polygon_area
FROM polygon_table